# Simple Linear Regression - Deep Explainer

**Algorithm:** Ordinary Least Squares (OLS) simple linear regression.
**Dataset:** 6 pizza records `(diameter_inches, price_usd)`.
**Goal of this notebook:** Show *every* arithmetic step, the closed-form formula, a tabular trace, and diagnostic plots so the model is fully transparent.

The model learns a straight line:

$$\hat{y} = m x + b$$

where `m` is the **slope** and `b` is the **intercept**.


## The Mathematics

We minimise the sum of squared residuals. The closed-form OLS solution is:

$$m = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}, \qquad b = \bar{y} - m\bar{x}$$

**Cost function (Mean Squared Error):**

$$J(m,b) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

**Goodness of fit (R-squared):**

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

This notebook fits the line **cumulatively** - first on 2 points, then 3, ... up to all 6 - so you can watch the parameters `m` and `b` converge.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Dataset: Pizza Diameters (inches) and Prices ($)
X = np.array([6, 8, 10, 12, 14, 18], dtype=float)
Y = np.array([7.0, 9.0, 11.5, 14.0, 16.5, 19.5], dtype=float)


class DetailedCumulativeRegressor:

  def __init__(self, initial_slope=0.5, initial_intercept=2.0):
    self.history_m = [initial_slope]
    self.history_b = [initial_intercept]
    self.labels = ["Initial Guess"]
    self.history_lines = [initial_slope * X + initial_intercept]

    # Structure to hold summary rows for our 12-column Matplotlib visual table
    self.summary_table_data = [[
        "Initial Guess",
        "-",
        "-",
        "-",
        "-",
        "-",
        "-",
        "-",
        "-",
        "-",
        f"{initial_slope:.3f}",
        f"{initial_intercept:.3f}",
    ]]

  def fit_and_print_details(self, x_data, y_data):
    print("=" * 155)
    print(
        "                                 DETAILED POINT-BY-POINT ORDINARY"
        " LEAST SQUARES CALCULATIONS"
    )
    print("=" * 155)

    for i in range(2, len(x_data) + 1):
      x_sub = x_data[:i]
      y_sub = y_data[:i]

      x_mean = np.mean(x_sub)
      y_mean = np.mean(y_sub)

      print(f"\n👉 [ DATASET STEP: First {i} Points ]")

      # Updated 12-column text header layout to perfectly mirror the requested layout
      print(
          f"   {'Dataset Iter.':<15} | {'Latest X':<8} | {'Latest Y':<8} |"
          f" {'Mean X':<8} | {'Mean Y':<8} | {'Dev. X':<8} | {'Dev. Y':<8} |"
          f" {'Prod. Dev.':<12} | {'Sum Prod (Num)':<15} | {'Sq. Dev (Den)':<14}"
          f" | {'Slope (m)':<10} | {'Intercept (b)':<13}"
      )
      print(f"   {'-'*151}")

      total_numerator = 0.0
      total_denominator = 0.0

      for idx in range(len(x_sub)):
        x_val = x_sub[idx]
        y_val = y_sub[idx]

        dx = x_val - x_mean
        dy = y_val - y_mean
        num_item = dx * dy
        den_item = dx**2

        total_numerator += num_item
        total_denominator += den_item

        # Print incremental metrics tracking up to the current coordinate step iteration
        iter_label = f"First {i} Pts" if idx == len(x_sub) - 1 else ""
        print(
            f"   {iter_label:<15} | {x_val:<8.1f} | {y_val:<8.1f} |"
            f" {x_mean:<8.2f} | {y_mean:<8.2f} | {dx:<8.2f} | {dy:<8.2f} |"
            f" {num_item:<12.2f} | {'-':<15} | {'-':<14} | {'-':<10} |"
            f" {'-':<13}"
        )

      m = total_numerator / total_denominator if total_denominator != 0 else 0
      b = y_mean - m * x_mean

      # Final row displays calculated summation outputs along with OLS parameter updates
      print(
          f"   {'📊 Final Row':<15} | {x_val:<8.1f} | {y_val:<8.1f} |"
          f" {x_mean:<8.2f} | {y_mean:<8.2f} | {dx:<8.2f} | {dy:<8.2f} |"
          f" {num_item:<12.2f} | {total_numerator:<15.2f} |"
          f" {total_denominator:<14.2f} | {m:<10.3f} | {b:<13.3f}"
      )
      print(f"   {'-'*151}")

      self.history_m.append(m)
      self.history_b.append(b)
      self.labels.append(f"First {i} Points")
      self.history_lines.append(m * X + b)

      # Append metrics matching the 12-column header format tracking the latest updates
      self.summary_table_data.append([
          f"First {i} Points",
          f"{x_val:.1f}",
          f"{y_val:.1f}",
          f"{x_mean:.2f}",
          f"{y_mean:.2f}",
          f"{dx:.2f}",
          f"{dy:.2f}",
          f"{num_item:.2f}",
          f"{total_numerator:.2f}",
          f"{total_denominator:.2f}",
          f"{m:.3f}",
          f"{b:.3f}",
      ])


# 2. Run Cumulative Detailed Fit Execution
model = DetailedCumulativeRegressor(initial_slope=0.5, initial_intercept=2.0)
model.fit_and_print_details(X, Y)

# 3. Plotting Setup (Grid Layout: 2 Rows, 4 Columns for Plots Below)
fig = plt.figure(figsize=(26, 12))  # Marginally widened canvas to hold the extensive header math comfortably
grid = fig.add_gridspec(2, 4, height_ratios=[1.2, 1.2], hspace=0.5, wspace=0.35)

# --- TOP ROW: UPDATED 12-COLUMN DATA BOARD WITH MATHEMATICAL FORMULAS ---
ax_table = fig.add_subplot(grid[0, :])
ax_table.axis("off")

# Your header layout containing explicit formula extensions using raw string syntax
headers = [
    "Dataset\nIteration",
    "Latest X\n($X_i$)",
    "Latest Y\n($Y_i$)",
    "Mean X\n($\\bar{X} = \\frac{\\sum X}{n}$)",
    "Mean Y\n($\\bar{Y} = \\frac{\\sum Y}{n}$)",
    "Deviations X\n($X_i - \\bar{X}$)",
    "Deviations Y\n($Y_i - \\bar{Y}$)",
    "Product of\nDeviations\n($(X_i - \\bar{X})(Y_i - \\bar{Y})$)",
    "Sum of Product\nof Deviations\n($\\sum (X - \\bar{X})(Y - \\bar{Y})$)(Num)",
    "Squared\nDeviations X\n($\\sum (X - \\bar{X})^2$)(Den)",
    "Slope (m)\n($m = \\frac{Num}{Den}$)",
    "Intercept (b)\n($b = \\bar{Y} - m\\bar{X}$)",
]

table = ax_table.table(
    cellText=model.summary_table_data,
    colLabels=headers,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)  # Set slightly smaller size to contain LaTeX formatting clean
table.scale(1, 2.6)  # Height expanded to offer vertical clearance for nested fractions

for col_idx in range(len(headers)):
  cell = table[0, col_idx]
  cell.set_facecolor("#2C3E50")
  cell.get_text().set_color("white")
  cell.get_text().set_weight("bold")

ax_table.set_title(
    "Incremental Ordinary Least Squares (OLS) Step Summary Matrix with Formula Layouts",
    fontsize=14,
    weight="bold",
    pad=15,
)

# --- BOTTOM ROW: COMPARISON PLOTS ---

# Plot 1: Regression Line Progression
ax1 = fig.add_subplot(grid[1, 0])
ax1.scatter(X, Y, color="red", label="Actual Data Points", s=80, zorder=5)
ax1.plot(
    X,
    model.history_lines[0],
    color="gray",
    linestyle="--",
    alpha=0.7,
    label="Initial Guess",
)

colors = ["#FFC107", "#FF9800", "#FF5722", "#E91E63"]
for idx, line in enumerate(model.history_lines[1:-1], start=1):
  ax1.plot(
      X,
      line,
      color=colors[idx - 1],
      linestyle=":",
      alpha=0.6,
      label=f"{model.labels[idx]}",
  )

ax1.plot(
    X,
    model.history_lines[-1],
    color="blue",
    linewidth=2.5,
    label="Final OLS Line",
)
ax1.set_title("Line Progression", fontsize=11, weight="bold")
ax1.set_xlabel("Diameter (inches)")
ax1.set_ylabel("Price ($)")
ax1.legend(fontsize=8, loc="upper left")
ax1.grid(True, linestyle="--", alpha=0.4)

# Plot 2: Slope History Tracking
ax2 = fig.add_subplot(grid[1, 1])
ax2.plot(
    model.labels,
    model.history_m,
    marker="o",
    color="purple",
    linewidth=1.8,
    label="Slope (m)",
)
for i, val in enumerate(model.history_m):
  ax2.annotate(
      f"{val:.2f}",
      (model.labels[i], model.history_m[i]),
      textcoords="offset points",
      xytext=(0, 8),
      ha="center",
      fontsize=9,
  )
ax2.set_title("Slope (m) Change", fontsize=11, weight="bold")
ax2.set_xticks(range(len(model.labels)))
ax2.set_xticklabels(model.labels, rotation=30, ha="right", fontsize=8.5)
ax2.grid(True, linestyle="--", alpha=0.4)

# Plot 3: Intercept History Tracking
ax3 = fig.add_subplot(grid[1, 2])
ax3.plot(
    model.labels,
    model.history_b,
    marker="s",
    color="green",
    linewidth=1.8,
    label="Intercept (b)",
)
for i, val in enumerate(model.history_b):
  ax3.annotate(
      f"{val:.2f}",
      (model.labels[i], model.history_b[i]),
      textcoords="offset points",
      xytext=(0, 8),
      ha="center",
      fontsize=9,
  )
ax3.set_title("Intercept (b) Change", fontsize=11, weight="bold")
ax3.set_xticks(range(len(model.labels)))
ax3.set_xticklabels(model.labels, rotation=30, ha="right", fontsize=8.5)
ax3.grid(True, linestyle="--", alpha=0.4)

# Plot 4: Mathematical Formula Reference Graph
ax4 = fig.add_subplot(grid[1, 3])
ax4.set_facecolor("#FDFEFE")

final_m, final_b = model.history_m[-1], model.history_b[-1]
ax4.plot(
    X,
    final_m * X + final_b,
    color="blue",
    linewidth=2,
    label="ŷ = mX + b",
    zorder=2,
)
ax4.scatter(X, Y, color="red", s=60, zorder=3)

mx, my = np.mean(X), np.mean(Y)
ax4.axvline(
    mx,
    color="darkorange",
    linestyle="--",
    linewidth=1.2,
    alpha=0.7,
    label=f"Mean X ({mx:.1f})",
)
ax4.axhline(
    my,
    color="teal",
    linestyle="--",
    linewidth=1.2,
    alpha=0.7,
    label=f"Mean Y ({my:.1f})",
)
ax4.scatter([mx], [my], color="black", s=100, marker="X", zorder=4)

formula_text = (
    r"$\mathbf{Linear\ Regression\ Formulas:}$"
    "\n\n"
    r"$\mathbf{1.\ Prediction\ Equation:}$"
    "\n"
    r"   $\hat{y} = mX + b$"
    "\n\n"
    r"$\mathbf{2.\ Slope\ (m):}$"
    "\n"
    r"   $m = \frac{\sum (X - \bar{X})(Y - \bar{Y})}{\sum (X - \bar{X})^2}$"
    "\n\n"
    r"$\mathbf{3.\ Intercept\ (b):}$"
    "\n"
    r"   $b = \bar{Y} - m\bar{X}$"
)

ax4.text(
    0.05,
    0.95,
    formula_text,
    transform=ax4.transAxes,
    fontsize=10.5,
    verticalalignment="top",
)
ax4.set_title("Formula Reference", fontsize=11, weight="bold")
ax4.legend(fontsize=8, loc="lower right")
ax4.grid(True, linestyle="--", alpha=0.4)

plt.show()


## Table Representation (pandas)

Below is the same 12-column trace produced during training, rendered as a clean `DataFrame` for inspection.


In [ ]:
import pandas as pd

linear_headers = [
    "Dataset Iter", "Latest X", "Latest Y", "Mean X", "Mean Y",
    "Dev X", "Dev Y", "Prod Dev", "SumProd Num", "SqDev Den", "Slope m", "Intercept b",
]
summary_df = pd.DataFrame(model.summary_table_data, columns=linear_headers)
print("=== OLS STEP-BY-STEP TABLE ===")
print(summary_df.to_string(index=False))


## Performance Metrics & Residual Analysis


In [ ]:
import pandas as pd
final_m, final_b = model.history_m[-1], model.history_b[-1]
y_pred_final = final_m * X + final_b
residuals = Y - y_pred_final

mse = np.mean(residuals**2)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((Y - np.mean(Y))**2)
r2 = 1 - ss_res / ss_tot

print("=== METRICS (final line on all 6 points) ===")
print(f"Final slope   (m) : {final_m:.4f}")
print(f"Final intercept(b) : {final_b:.4f}")
print(f"Mean Squared Error : {mse:.4f}")
print(f"R-squared (R2)     : {r2:.4f}")

# Per-point residual table
res_df = pd.DataFrame({
    "X (diameter)": X,
    "Y actual": Y,
    "Y predicted": np.round(y_pred_final, 3),
    "residual": np.round(residuals, 3),
})
print()
print("=== PER-POINT RESIDUALS ===")
print(res_df.to_string(index=False))


## Key Takeaways

- **`m` (slope)** tells us price increases by ~`m` dollars per extra inch of diameter.
- **`b` (intercept)** is the theoretical price at 0 inches (often non-physical, but mathematically required).
- As we add more points the OLS estimates **stabilise** - visible in the slope/intercept history plots above.
- **R2 close to 1** means the straight line explains almost all variance in this (near-linear) dataset.
